# OOP Part 2

## Encapsulation; Copying Objects; Dynamic Memory Allocation of Objects

<br><br><br><br><br>
<br><br><br><br><br>

# 1. Encapsulation

### * In the last Mario example, the *client* code never makes reference to the attribute variables `m.name_`, `m.coins_`, and `m.lives_`.  

### * Instead, the creation, accessing and changing of these values for an object are done via **methods**.  

### * This is desirable!  As the designer of a class, your goal is to design an interface (that is, a set of methods) that is strong enough to capture all the things a client might want to do with objects. To put it starkly:

    Code inside a class *definition* can reference attribute variables.
    
    *Client* code should try to NOT reference attribute variables -- it ought to only interact with ENTIRE objects, via INTERFACE METHODS.

### * This idea is known as *encapsulation*.   The examples with methods are "better", because they don't violate encapsulation.  

<br><br><br><br><br>
<br><br><br><br><br>

### * Why is this important?  When you BUILD a car, the technical bits under the hood are important, but when you DRIVE the car, the technical bits are distracting at best, harmful at worst.

### * The attribute variables are frequently thought of as technical details used to represent the object.  The user isn't meant to directly look at or modify them; instead, they ought to interact with them through the interface -- that is, the methods -- which (if well-designed) are guaranteed to access or modify objects in appropriate ways.

### * In fact, that's what the underscores signify.   They are a custom, meant to convey to other programmers who are using the class: "hey, I am a technical detail! Don't reference me directly! If you *do* use me you'll probably screw something up, and it's your own fault then!  Be smart and use the interface instead!"

<br><br><br><br><br>
<br><br><br><br><br>


### * When client code *uses* the class, it usually won't refer directly to attribute variables of objects, but rather it will instead use methods to change/access those objects. 


### * I.e. should do less of this:

`x.attribute_ = value`

### * and more of this: 

`x.set_attribute(value)`

### * "The attribute variables are the technical underside, while the methods are the usable interface" -- that's what those underscores signify.

### * Ok, but *why is that important*? Is it just a matter of taste? NO! Two reasons. But first, a quick exercise.

<br><br><br><br><br>
<br><br><br><br><br>

### YOU TRY: The Employee class below has client code that directly modifies `salary_` to give a raise. Rewrite this so that the class has a `giveRaise()` method instead, and the client code uses that method.

<br><br><br><br><br>
<br><br><br><br><br>

In [ ]:
////employeeEncapsulation.cpp
// Example 1a: Employee without encapsulation
#include<iostream>
#include<string>
class Employee{
    public:
    std::string name_; std::string role_; double salary_;
    Employee(std::string n, std::string r){
        this->name_ = n;
        this->role_ = r;
        this->salary_ = 10000;
    }
    // TODO: write a giveRaise() method that takes a percentage
    // and increases salary_ by that percent
    void  display(){

        std::cout << e.name_ << " earns " << this->salary_ << "\n";
    }



};
int main(){
    Employee e("Areeba", "Math Lecturer");
    // This directly modifies the attribute -- not great!
    e.salary_ *= 1.15;
    std::cout << e.name_ << " now earns " << e.salary_ << "\n";
    // TODO: rewrite the line above to use your giveRaise() method instead
    e.display();

}

<br><br><br><br><br>
<br><br><br><br><br>

### Now, let's look at two reasons why encapsulation is important.

### **1.** (The less important reason) *Interacting with objects via methods ensures that clients interact with objects CORRECTLY.*

In [ ]:
//// badClientCode.cpp
// Example 1b: Drawing cards, badly
//% CXXFLAGS -std=c++11 
#include <iostream>
#include <vector>
#include <cstdlib> // for random number generation via rand() -- although this isn't the best way to do it
#include <ctime> // setting the seed for rand so that different values are generated
#include "areebas_card_v1.cpp" // contains the definitions for Card and Deck - remember, really would want .h file instead

int main() {
    srand(static_cast<unsigned int>(time(0))); // Seed for random number generation
    Deck d; // Create a deck.

    // Pick 10 cards, in a flagrantly wrong way.
    for (int i = 0; i < 10; ++i) {
        int card_num = rand() % d._cards.size(); // Look at the length of the deck, and pick an index in that range
        Card current_card = d._cards[card_num]; // Pick the card in that index
        current_card.display(); // Display it
    }

    return 0;
}

In [ ]:
//// badClientCode.cpp
// Example 1b': Drawing cards, badly - with removal of card
//% CXXFLAGS -std=c++11 
#include <iostream>
#include <vector>
#include <cstdlib>
#include <ctime>
#include "areebas_card_v1.cpp" // contains the definitions for Card and Deck

int main() {
    srand(static_cast<unsigned int>(time(0))); // Seed for random number generation
    Deck d; // Create a deck.

    // Pick 10 cards, in a flagrantly wrong way.
    for (int i = 0; i < 10; ++i) {
        int card_num = rand() % d._cards.size(); // Look at the length of the deck, and pick an index in that range
        Card current_card = d._cards[card_num]; // Pick the card in that index
        d._cards.erase(d._cards.begin() + card_num); // Remove the card from the deck 
        current_card.display(); // Display it
    }

    return 0;
}

In [ ]:
////morebadClientcode.cpp
// Example 1b'': better way to do random, plus this code actually deletes the card that is picked
//% CXXFLAGS -std=c++11 

#include <iostream>
#include <vector>
#include <random>
#include "areebas_card_v1.cpp" // contains the definitions for Card and Deck
int main() {
    Deck d; // Create a deck. 
    std::random_device rd; // Obtain a random number from hardware
    std::mt19937 eng(rd()); // Seed the generator
    

    for (int i = 0; i < 10; ++i) { // This references the member variable ._cards
         // and it WILL be broken after a change in implementation.
        std::uniform_int_distribution<> distr(0, d._cards.size() - 1); // Define the range
        int card_num = distr(eng);
        d._cards[card_num].display();
        d._cards.erase(d._cards.begin() + card_num); // Remove the card from the deck 
    }
}

### * The problem here is that none of the code in the first loop actually takes care of the part where cards are *removed* from the deck after they are picked.  

### * Maybe that's a dumb mistake -- but the more complex your classes get, the more difficult it gets to avoid similar mistakes.  

### * Sticking to well-designed methods is a way to avoid such mistakes.

<br><br><br><br><br>
<br><br><br><br><br>


### **2.** (The more important reason) *Interacting with objects via methods ensures that your client code won't be broken by IMPLEMENTATION CHANGES.*

### * Imagine that, after I had already started using my class, I decided to change how I represented a `Deck`. Instead of using a list of `Card`s which we shuffle at the time of initialization, suppose that I decided that it was "more efficient" to represent a `Deck` as a 4 x 13 vector, containing boolean values: the rows will correspond to all the possible suits, the columns will correspond to all the possible ranks, and the values will be `True` if the corresponding card has already been drawn, and `False` if it has not been drawn.  Whatever.

### * The lines of code that **only reference methods** will still work if we use the updated class; the code that references attributes will break!

### * As a simpler example: remember the `Employee` class? What if we decided to store salary as an `int` in *cents* instead of a `double` in dollars, to avoid floating-point precision issues? Client code that does `e.salary_ *= 1.15` would break -- `salary_` doesn't exist anymore! But `e.giveRaise(15)` would still work perfectly, because the method handles the internal details.

In [ ]:
////moreClientcode.cpp
// Example 1c: Changing Implementation
//% CXXFLAGS -std=c++11 

#include <iostream>
#include <vector>
#include <cstdlib> // included for using rand()
#include <ctime>   // included for setting the seed for rand()
#include "areebas_card_v2.cpp" // contains the definitions for Card and Deck

int main() {
    Deck d; // Create a deck.

    std::cout << "The good loop:\n" << std::endl;
    for (int i = 0; i < 10; ++i) { // This only uses methods.
        Card c = d.draw();    // It will NOT be broken by a change in implementation.
        c.display();          // 
    }

    std::cout << "\nThe bad loop:\n" << std::endl; 
    srand(static_cast<unsigned int>(time(0))); 
    for (int i = 0; i < 10; ++i) {
        int card_num = rand() % d._cards.size(); // This references the member variable ._cards
                                                 // and it WILL be broken after a change in implementation.
        d._cards[card_num].display(); // Display it
        d._cards.erase(d._cards.begin() + card_num); // Remove the card from the deck 
    }

    return 0;
}

In [ ]:
////moreClientcode.cpp
// Example 1c': Changing Implementation - good loop only
//% CXXFLAGS -std=c++11 

#include <iostream>
#include <vector>
#include <cstdlib> // included for using rand()
#include <ctime>   // included for setting the seed for rand()
#include "areebas_card_v2.cpp" // contains the definitions for Card and Deck

int main() {
    Deck d; // Create a deck.

    std::cout << "The good loop:\n" << std::endl;
    for (int i = 0; i < 10; ++i) { // This only uses methods.
        Card c = d.draw();    // It will NOT be broken by a change in implementation.
        c.display();          // 
    }

    return 0;
}

### * So, as a client, respecting the so-called "privacy" of attribute variables means that your code will be much less likely to break in the event of implementation change! 

### * And as a class designer, you can *refactor* your class safely so that it works better or more efficiently, as long as you don't change the way the interface works.

<br><br><br><br><br>
<br><br><br><br><br>
<br><br><br><br><br>
<br><br><br><br><br>

# 2. Copying Objects

### * Last class, some of you wrote code like this to track the highest card:

```cpp
Card highest = card1;
if (card2.beats(highest)) highest = card2;
if (card3.beats(highest)) highest = card3;
highest.display();
```

### * This works! But what exactly is happening when we write `Card highest = card1;`? Is `highest` the same object as `card1`, or a separate copy?

### * Let's investigate using the `Character` class.

<br><br><br><br><br>
<br><br><br><br><br>

In [ ]:
////copyingObjects.cpp
// Example 2a: Copying objects -- are they the same object?
#include<iostream>
#include<string>
class Character{
   public: std::string name_; int lives_, coins_;
   Character(std::string n){
        this->name_ = n; this->lives_ = 3; this->coins_ = 0;
   }
   void display() const {
        std::cout << this->name_ << ": " << this->lives_ << " lives, " << this->coins_ << " coins\n";
   }
   void collect_coins(int num_coins){
       this->coins_ += num_coins;
   }
};
int main(){
    // Create a Character m

    // Copy m into m2

    // Check: are they at the same address?


    // Modify m2 and see if m changes


    // Display both


}

### * They are at **different addresses** -- `m2` is a completely independent object! Modifying `m2` does not affect `m`.

### * When we write `Character m2 = m;`, the compiler calls the **copy constructor**. Since we didn't write one, the compiler generates a **default copy constructor** that copies each member variable one by one: `m2.name_` gets a copy of `m.name_`, `m2.lives_` gets a copy of `m.lives_`, etc.

### * Compare this to a pointer approach:

```cpp
Character* ptr = &m;  // ptr POINTS to m -- no copy, same object
ptr->collect_coins(100);
m.display();  // m now has 100 coins!
```

### * With the pointer, changes through `ptr` affect `m` because they refer to the same object. With the copy, `m2` is independent.

<br><br><br><br><br>
<br><br><br><br><br>

### * We can write our own copy constructor to see exactly when copying happens. The copy constructor takes a `const` reference to another object of the same class.
### * What is a `const` reference? Let's not worry about that for now. This is a preview and we'll get into the details later.

In [ ]:
////copyConstructor.cpp
// Example 2b: Writing our own copy constructor
#include<iostream>
#include<string>
class Character{
   public: std::string name_; int lives_, coins_;
   Character(std::string n){
        this->name_ = n; this->lives_ = 3; this->coins_ = 0;
        std::cout << "Constructor called for " << n << "\n";
   }
   // Copy constructor -- takes a const reference to another object of the same class
   Character(const Character& other){
        // Print a message and copy each member from other




   }
   void display() const {
        std::cout << this->name_ << ": " << this->lives_ << " lives, " << this->coins_ << " coins\n";
   }
   void collect_coins(int num_coins){
       this->coins_ += num_coins;
   }
};
int main(){
    Character m("Mario");
    m.collect_coins(50);

    // Two ways to call the copy constructor:



}

### * Notice that both `Character m2 = m;` and `Character m3(m);` call the copy constructor. They are two different syntaxes for the same thing.

### * The default copy constructor that the compiler generates does exactly what our explicit one does -- it copies each member using that member's own copy constructor. For `int`, `string`, `vector`, etc., this produces a completely independent deep copy. 

### * But for **raw pointers** (`int*`, `char*`, etc.), copying just copies the address -- NOT the data it points to. This is where things can go wrong, and we'll see why in section 4.

<br><br><br><br><br>
<br><br><br><br><br>

### YOU TRY: Consider the `Character` class from above (with `name_`, `lives_`, and `coins_`). Read through the code below. Draw a memory diagram showing both objects on the stack after all the code has executed, and predict the output.

<br><br><br><br><br>
<br><br><br><br><br>

In [ ]:
////copyYouTry.cpp
// Example 2c: You try -- predict the output!
#include<iostream>
#include<string>
class Character{
   public: std::string name_; int lives_, coins_;
   Character(std::string n){
        this->name_ = n; this->lives_ = 3; this->coins_ = 0;
   }
   void display() const {
        std::cout << this->name_ << ": " << this->lives_ << " lives, " << this->coins_ << " coins\n";
   }
   void collect_coins(int num_coins){
       this->coins_ += num_coins;
   }
   bool die(){
       this->lives_ -= 1;
       return this->lives_ > 0;
   }
};
int main(){
    Character m("Mario");
    m.collect_coins(50);
    Character m2 = m;
    m2.name_ = "Wario";
    m2.collect_coins(25);
    m.die();
    m.display();
    m2.display();
}


<br><br><br><br><br>
<br><br><br><br><br>
<br><br><br><br><br>
<br><br><br><br><br>

# 3. Dynamic Memory Allocation of Objects

### * We've already seen we can allocate memory for objects on the heap by using the keyword `new` (and then deallocate using `delete`). We can do this now with objects of classes that we define too.

### * Notice that in this version of the `Employee` class, we've made the attributes `private` -- this enforces encapsulation! Client code can only interact with Employee objects through the public methods.

<br><br><br><br><br>
<br><br><br><br><br>

In [ ]:
////employeeDynamicMemory.cpp
// Example 3a: Dynamic Memory allocation of objects
#include<iostream>
#include<string>
class Employee{
    private:
        std::string name_; std::string role_; double salary_; 
    public:
    Employee(std::string n, std::string r){
        this->name_ = n;
        this->role_ = r;
        this->salary_ = 10000;
    }
    void giveRaise(double percent){
        this->salary_ *= (1 + percent/100);
    }
    void display() const { // const is used for accessor methods
        std::cout << this->name_ << " has role " << this->role_ << " and salary " << this->salary_ << "\n";
    }
};
int main(){
    // Allocate an Employee on the heap using new

    //e.display(); // this won't work -- e is a pointer, not an Employee
    // Dereference first, then call the method

    // Arrow operator shorthand

    // same thing for attributes: e->role_, not e.role_
    // Give a raise and display


    // Free the memory

}

### * When we use `new`, the object is created on the **heap** instead of the stack. The variable `e` is a pointer that lives on the stack but points to the object on the heap.

### * Since `e` is a pointer, we can't use `e.display()` -- we need to dereference first: `(*e).display()`. The **arrow operator** `e->display()` is shorthand for this.

### * And since we used `new`, we are responsible for calling `delete` to free the memory when we're done!

<br><br><br><br><br>
<br><br><br><br><br>

### * What happens if we forget to `delete`? The object stays on the heap even after the program ends -- this is a **memory leak**. We can use the **destructor** (a special method called `~ClassName()`) to see when an object is destroyed.

In [ ]:
////employeeDynamicMemory.cpp
// Example 3a': Dynamic Memory allocation - with destructor to track cleanup
#include<iostream>
#include<string>
class Employee{
    private:
        std::string name_; std::string role_; double salary_; 
    public:
    Employee(std::string n, std::string r){
        this->name_ = n;
        this->role_ = r;
        this->salary_ = 10000;
    }
    void giveRaise(double percent){
        this->salary_ *= (1 + percent/100);
    }
    void display() const {
        std::cout << this->name_ << " has role " << this->role_ << " and salary " << this->salary_ << "\n";
    }
    // Write a destructor



};
int main(){
    Employee* e = new Employee("Areeba", "Math Lecturer");
    e->display();
    e->giveRaise(15);
    e->display();
    // delete e; // try commenting/uncommenting this line -- does the destructor get called?
}

<br><br><br><br><br>
<br><br><br><br><br>

### * In practice, whenever possible, use **smart pointers** to avoid common bugs associated with manual memory allocation. A `std::unique_ptr` automatically calls `delete` for you when the pointer goes out of scope -- no memory leaks!

In [ ]:
////employeeDynamicMemory.cpp
//% CXXFLAGS -std=c++17
// Example 3b: Dynamic Memory allocation of objects using smart pointers
#include<iostream>
#include<string>
#include<memory>
class Employee{
    private:
        std::string name_; std::string role_; double salary_; 
    public:
    Employee(std::string n, std::string r){
        this->name_ = n;
        this->role_ = r;
        this->salary_ = 10000;
    }
    void giveRaise(double percent){
        this->salary_ *= (1 + percent/100);
    }
    void display() const {
        std::cout << this->name_ << " has role " << this->role_ << " and salary " << this->salary_ << "\n";
    }
    ~Employee(){
        std::cout << "Employee destructor is called\n";
    }
};
int main(){
    // Create a unique_ptr to an Employee

    // Call display using the arrow operator

    // No need to call delete -- the smart pointer handles it automatically!
    // Notice the destructor IS called even though we never wrote delete.

}

<br><br><br><br><br>
<br><br><br><br><br>

### YOU TRY: Using the `Character` class (with `name_`, `lives_`, `coins_`, and methods `collect_coins()`, `die()`, `display()`), create a `Character` object on the heap.  Give the character 100 coins using the arrow operator, display the character, and then free the memory.

<br><br><br><br><br>
<br><br><br><br><br>

In [ ]:
////characterHeap.cpp
// Example 3c: You try -- Character on the heap
#include<iostream>
#include<string>
class Character{
   public: std::string name_; int lives_, coins_;
   Character(std::string n){
        this->name_ = n; this->lives_ = 3; this->coins_ = 0;
   }
   void display() const {
        std::cout << this->name_ << ": " << this->lives_ << " lives, " << this->coins_ << " coins\n";
   }
   void collect_coins(int num_coins){
       this->coins_ += num_coins;
   }
   bool die(){
       this->lives_ -= 1;
       return this->lives_ > 0;
   }
};
int main(){
    // Create a Character on the heap, give it 100 coins, display it, and free the memory




}

# Summary

### * **Encapsulation**: Client code should interact with objects through methods, not by directly accessing attribute variables. This ensures correct usage and protects against implementation changes.

### * **Copying Objects**: When you assign one object to another (`Character m2 = m;`), the compiler calls the **copy constructor**, which copies each member variable. The result is a completely independent object. The default copy constructor works well for simple types (`int`, `string`, `vector`) -- but we'll see later that it can cause problems when a class manages dynamic memory with raw pointers.

### * **Dynamic Memory**: Objects can be allocated on the heap using `new`. Use the **arrow operator** (`->`) to call methods on a pointer to an object. Always `delete` heap-allocated objects, or use **smart pointers** (`std::unique_ptr`) to handle cleanup automatically.

### * Next time: What happens when the default copy constructor goes wrong? The **Rule of Three** (destructor, copy constructor, copy assignment operator), and more.